# 06 — Baseline Classifier: Keypoints → Sign

## 1. Learning Goal

Build the **simplest possible** sign classifier and measure it honestly.

> The goal is not maximum accuracy. The goal is a **baseline**: a reference number that every fancier model later must beat.

You will:

1. Turn every image into a **feature vector** (40 numbers from the hand keypoints)
2. Train **KNN** and **Logistic Regression** with scikit-learn
3. Compare them with a **do-nothing baseline**
4. Read a **confusion matrix** and a classification report
5. Test whether **normalization** really helps
6. Look at the **mistakes**

## 2. Concept

```text
Image ─► MediaPipe ─► 21 keypoints ─► normalize ─► 40 numbers ─► classifier ─► sign
        (notebook 05)                  (notebook 05)               (this notebook)
```

```text
 K-Nearest Neighbours (KNN):  "find the K most similar training examples; majority vote"
 Logistic Regression:         "learn one weight per feature per class, turn the score into probabilities"
```

## 3. Why do we need this?

Without a baseline you cannot tell whether a CNN or Transformer is actually better. Often the simple model is surprisingly good, and **knowing that is valuable**.

In [ ]:
import time
import cv2
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# Works whether you start Jupyter in the project folder or inside notebooks/
ROOT = Path("..") if Path("../data").exists() else Path(".")
DATA = ROOT / "data"
MODELS = ROOT / "models"
DATA.mkdir(exist_ok=True)
MODELS.mkdir(exist_ok=True)

def show_bgr(img, title="", size=6):
    plt.figure(figsize=(size, size))
    plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))   # OpenCV is BGR, Matplotlib wants RGB
    plt.title(title)
    plt.axis("off")
    plt.show()

def describe(image, name="image"):
    print(f"--- {name} ---")
    print("shape :", image.shape, "| dtype:", image.dtype, "| min:", image.min(), "| max:", image.max())

In [ ]:
import mediapipe as mp

def create_detector(running_mode="image", num_hands=1, min_detection=0.5):
    """Load the MediaPipe Hand Landmarker model file from models/."""
    modes = {"image": mp.tasks.vision.RunningMode.IMAGE,
             "video": mp.tasks.vision.RunningMode.VIDEO}
    options = mp.tasks.vision.HandLandmarkerOptions(
        base_options=mp.tasks.BaseOptions(model_asset_path=str(MODELS / "hand_landmarker.task")),
        running_mode=modes[running_mode],
        num_hands=num_hands,
        min_hand_detection_confidence=min_detection,
    )
    return mp.tasks.vision.HandLandmarker.create_from_options(options)

def detect_hand(detector, bgr_image):
    """OpenCV gives BGR, MediaPipe wants RGB. Convert, wrap, detect."""
    rgb = cv2.cvtColor(bgr_image, cv2.COLOR_BGR2RGB)
    mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)
    return detector.detect(mp_image)

def landmarks_to_array(result, hand_index=0):
    """result -> NumPy array of shape (21, 3): columns are x, y, z (x and y are fractions 0..1)."""
    hand = result.hand_landmarks[hand_index]
    return np.array([[lm.x, lm.y, lm.z] for lm in hand], dtype=np.float32)

In [ ]:
import pandas as pd
import joblib
from sklearn.dummy import DummyClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, ConfusionMatrixDisplay

---
# Part 1 — Images → keypoint table

We use `data/splits.csv` made in notebook 03 (columns: `path, label, session, split`). For every image we run the hand detector and store the 21 `(x, y)` pixel positions. Images with **no detected hand** are counted and dropped.

This takes a little while the first time. The result is cached in `data/processed/keypoints.csv`, so re-running is instant.

In [ ]:
COLS = [f"{axis}{i}" for i in range(21) for axis in "xy"]        # x0, y0, x1, y1, ... x20, y20
KP_CSV = DATA / "processed" / "keypoints.csv"
KP_CSV.parent.mkdir(exist_ok=True)

if KP_CSV.exists():
    kp = pd.read_csv(KP_CSV)
    print("loaded cached keypoints:", kp.shape)
else:
    splits = pd.read_csv(DATA / "splits.csv")
    det = create_detector()
    rows, failed = [], []
    for _, row in splits.iterrows():
        image = cv2.imread(row.path)
        res = detect_hand(det, image)
        if not res.hand_landmarks:
            failed.append(row.label)
            continue
        h, w = image.shape[:2]
        xy = landmarks_to_array(res)[:, :2] * np.array([w, h])      # pixels
        rows.append({"path": row.path, "label": row.label, "session": row.session,
                     "split": row.split, **dict(zip(COLS, xy.flatten()))})
    kp = pd.DataFrame(rows)
    kp.to_csv(KP_CSV, index=False)
    print(f"hand found in {len(kp)} of {len(splits)} images; saved to {KP_CSV}")
    print("failures per class:", pd.Series(failed).value_counts().to_dict())

print(kp.groupby("split").size().to_dict())

### What to observe

- How many images had **no hand detected**? Are failures concentrated in one class? A sign the detector struggles with (for example a fist) is a **data problem** you would otherwise blame on the classifier.

---
# Part 2 — Features

The same two functions as notebook 05:

In [ ]:
def normalize_keypoints(xy):
    xy = np.asarray(xy, dtype=np.float32)
    centred = xy - xy[0]                           # wrist -> origin
    scale = np.linalg.norm(centred[9])             # hand size
    if scale < 1e-6:
        scale = 1.0
    return centred / scale

def to_features(xy):
    return normalize_keypoints(xy)[1:].flatten()   # 40 numbers

coords = kp[COLS].to_numpy(dtype=np.float32).reshape(-1, 21, 2)    # (N, 21, 2)
X_norm = np.array([to_features(c) for c in coords])                # (N, 40)  normalized
X_raw = coords.reshape(len(coords), -1)                            # (N, 42)  raw pixels
y = kp.label.to_numpy()
split = kp.split.to_numpy()

train, val, test = (split == "train"), (split == "val"), (split == "test")
print("X_norm:", X_norm.shape, "| X_raw:", X_raw.shape)
print("train / val / test sizes:", train.sum(), val.sum(), test.sum())

---
# Part 3 — Baselines: from "do nothing" to KNN and Logistic Regression

We always begin with a **dummy** classifier that always predicts the most common class. If a real model cannot beat it, something is wrong.

Models are trained on `train` and measured on `val`. **We do not touch `test` yet.**

In [ ]:
def fit_and_score(name, model, X):
    model.fit(X[train], y[train])
    acc_train = accuracy_score(y[train], model.predict(X[train]))
    acc_val = accuracy_score(y[val], model.predict(X[val]))
    print(f"{name:28s} train acc {acc_train:.3f}   val acc {acc_val:.3f}")
    return model, acc_val

dummy, _ = fit_and_score("Dummy (most frequent)", DummyClassifier(strategy="most_frequent"), X_norm)
knn, _ = fit_and_score("KNN (k=5)", KNeighborsClassifier(n_neighbors=5), X_norm)
logreg, _ = fit_and_score("Logistic Regression", LogisticRegression(max_iter=2000), X_norm)

### What to observe

- The dummy tells you the **floor**.
- Compare `train acc` with `val acc`. A big gap = the model memorised the training data (overfitting, which is the subject of notebook 08).
- Because validation is a **different session**, this is an honest number.

---
# Part 4 — Does normalization really help?  (Experiment EXP-004)

**Hypothesis:** with raw pixel coordinates, the classifier gets confused by where the hand is and how big it appears. Normalization should improve validation accuracy.

In [ ]:
print("Raw pixel coordinates vs normalized features")
fit_and_score("KNN raw", KNeighborsClassifier(n_neighbors=5), X_raw)
fit_and_score("KNN normalized", KNeighborsClassifier(n_neighbors=5), X_norm)
fit_and_score("LogReg raw", LogisticRegression(max_iter=5000), X_raw)
fit_and_score("LogReg normalized", LogisticRegression(max_iter=2000), X_norm)

### What to observe

Did the hypothesis hold? Write your result in the experiment log. If normalized is *not* better for your data, think about why. (Maybe you held your hand at the same distance every time?)

## Experiment: how many neighbours?

KNN's `k` is a **hyperparameter**: something we choose, not something learned.

In [ ]:
ks = [1, 3, 5, 9, 15, 25]
train_acc, val_acc = [], []
for k in ks:
    m = KNeighborsClassifier(n_neighbors=k).fit(X_norm[train], y[train])
    train_acc.append(accuracy_score(y[train], m.predict(X_norm[train])))
    val_acc.append(accuracy_score(y[val], m.predict(X_norm[val])))

plt.plot(ks, train_acc, "o-", label="train")
plt.plot(ks, val_acc, "s-", label="validation")
plt.xlabel("k (number of neighbours)"); plt.ylabel("accuracy"); plt.legend(); plt.grid(True)
plt.show()

### What to observe

With `k = 1`, train accuracy is 100% (each point is its own nearest neighbour), but validation is lower. That gap is overfitting made visible.

---
# Part 5 — Confusion matrix and the classification report

Accuracy is one number. **Which signs get mixed up?**

```text
              predicted
            A    B    L
 true  A  [ 18    1    0 ]     ← one A was called B
       B  [  0   19    0 ]
       L  [  2    0   17 ]     ← two L were called A
```

Reading it: rows = the truth, columns = the prediction. The diagonal is correct. Anything off the diagonal is a mistake, and tells you which signs look alike to the model.

In [ ]:
chosen = logreg                                  # change to knn and compare
pred_val = chosen.predict(X_norm[val])

ConfusionMatrixDisplay.from_predictions(y[val], pred_val, cmap="Blues", colorbar=False)
plt.title("Validation confusion matrix")
plt.show()

print(classification_report(y[val], pred_val, zero_division=0))

### What to observe

- Which pair of signs is confused most? Do they look similar to you?
- `precision` for a sign: when the model *says* this sign, how often is it right? `recall`: of all *real* examples of the sign, how many did it find? (Same ideas as in notebook 04.)

## Look at the actual mistakes

In [ ]:
val_idx = np.where(val)[0]
wrong = [i for i, p in zip(val_idx, pred_val) if p != y[i]]
print("mistakes:", len(wrong), "of", len(val_idx))

show = wrong[:8]
if show:
    fig, axes = plt.subplots(2, 4, figsize=(12, 6))
    for ax in axes.flat:
        ax.axis("off")
    for ax, i in zip(axes.flat, show):
        ax.imshow(cv2.cvtColor(cv2.imread(kp.path[i]), cv2.COLOR_BGR2RGB))
        ax.set_title(f"true {y[i]} -> pred {chosen.predict(X_norm[i:i+1])[0]}", fontsize=10)
    plt.tight_layout(); plt.show()

### What to observe

Are the mistakes **model** errors, or bad data (hand cut off, motion blur, wrong label, two signs that are genuinely similar mid-transition)? This is error analysis. It is often more useful than tuning the model.

---
# Part 6 — The final honest score: the test set (ONCE)

Pick your best model **using validation only**. Now evaluate on `test` a single time and write the number down. Do not tune anything afterwards.

In [ ]:
final_model = logreg                              # your choice, decided using validation
test_acc = accuracy_score(y[test], final_model.predict(X_norm[test]))
print(f"TEST accuracy: {test_acc:.3f}")

ConfusionMatrixDisplay.from_predictions(y[test], final_model.predict(X_norm[test]), cmap="Greens", colorbar=False)
plt.title("Test confusion matrix"); plt.show()

joblib.dump(final_model, MODELS / "baseline_keypoint_classifier.joblib")
print("saved", MODELS / "baseline_keypoint_classifier.joblib")

---
# Part 7 — Predict from a single new image (inference)

```text
Training  : model learns from labelled data
Validation: we compare models on unseen data
Inference : the finished model is used on brand-new input (what the final app does)
```

In [ ]:
def predict_sign(model, bgr_image, detector):
    res = detect_hand(detector, bgr_image)
    if not res.hand_landmarks:
        return None, 0.0
    h, w = bgr_image.shape[:2]
    xy = landmarks_to_array(res)[:, :2] * np.array([w, h])
    probs = model.predict_proba(to_features(xy).reshape(1, -1))[0]
    best = int(np.argmax(probs))
    return model.classes_[best], float(probs[best])

new_photo = DATA / "myhand.jpg"
if new_photo.exists():
    detector = create_detector()
    label, confidence = predict_sign(final_model, cv2.imread(str(new_photo)), detector)
    print("prediction:", label, "| confidence:", round(confidence, 3))

### Live version (optional)

Set `RUN_LIVE = True` for a webcam prediction. Press **q** to quit. We mirror the frame exactly like the training data.

In [ ]:
RUN_LIVE = False

if RUN_LIVE:
    live_detector = create_detector("video")
    cap = cv2.VideoCapture(0)
    start, last_ts = time.perf_counter(), -1
    while True:
        ok, frame = cap.read()
        if not ok:
            break
        frame = cv2.flip(frame, 1)
        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        ts = max(int((time.perf_counter() - start) * 1000), last_ts + 1)
        last_ts = ts
        res = live_detector.detect_for_video(mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb), ts)

        text = "no hand"
        if res.hand_landmarks:
            h, w = frame.shape[:2]
            xy = landmarks_to_array(res)[:, :2] * np.array([w, h])
            probs = final_model.predict_proba(to_features(xy).reshape(1, -1))[0]
            best = int(np.argmax(probs))
            text = f"{final_model.classes_[best]}  {probs[best]:.0%}"
        cv2.putText(frame, text, (10, 40), cv2.FONT_HERSHEY_SIMPLEX, 1.2, (0, 255, 0), 3)
        cv2.imshow("baseline (q to quit)", frame)
        if cv2.waitKey(1) & 0xFF == ord("q"):
            break
    cap.release()
    cv2.destroyAllWindows()

### What to observe in the live demo

Hold each sign. Does the label flicker? How does confidence behave between signs, or with no sign at all? The classifier only knows your classes, so it will name *something* even for a random hand shape. Think about how to handle "unknown" (threshold on confidence, an extra "none" class).

---
# 9. Results

```text
splits.csv ─► MediaPipe keypoints ─► normalize ─► 40 features ─► KNN / LogReg
                                                                    │
                                          Dummy baseline  <  KNN  ≈  LogReg
                                          validation (new session) → test (once)
```

# 10. What did we learn?

1. Features matter: **normalized keypoints** are a compact, background-free input.
2. Always compare against a **dummy baseline**.
3. Compare **train vs validation** accuracy to spot overfitting.
4. A **confusion matrix** shows *which* signs are confused; error analysis shows *why*.
5. **Test set once**, at the end.
6. The same pipeline (detect → features → predict) is used for **inference** in the live app.

# 11. Questions to think about

1. Why can Logistic Regression be a "linear" model yet separate hand shapes? What assumptions does it make about the 40 features?
2. Your classifier gives a confident answer for a hand shape that is *not* one of your signs. Why? How could you reject it?
3. If validation accuracy is 99% but test accuracy is 80%, what are two possible reasons?
4. Which signs would you expect keypoints alone to struggle with? (Think about signs that differ in **movement** or in **facial expression**.)

# 12. Mini challenge

1. Add **two more signs** and re-run notebooks 03 and 06. How does accuracy and the confusion matrix change?
2. Add the **z coordinate** as extra features. Does validation accuracy improve?
3. Add **angle features**: for each finger the angle at its middle joint. Is a hand-made feature better than raw coordinates?

---
## Experiment log

```text
EXP-004  normalization
Model    : KNN k=5, LogReg
Dataset  : my signs, split by session
Val acc  : raw =        normalized =
Conclusion :

EXP-005  baseline
Model    : LogReg on 40 normalized features
Train acc :     Val acc :     Test acc :
Biggest confusion :
Conclusion :
```

**Milestone 3 reached** when you have a classifier that recognises your signs on new data and you can read its confusion matrix.
**Next:** `07_neural_network_basics.ipynb`. We replace the black box by understanding a single neuron.